# Train the quantile transformer (any GPU notebook: Kaggle / Colab / Studio Lab)
Run all cells top to bottom. The setup cell clones the repo itself if it isn't already inside one, so a fresh Kaggle or Colab session needs no manual steps — on Kaggle, switch Session options to a GPU accelerator and turn **Internet** on first.

**Kaggle GPU choice: use T4 ×2, not the P100.** The P100 is a Pascal card (compute capability sm_60) that current PyTorch wheels no longer ship kernels for — training crashes at the first kernel launch. The T4 (sm_75) is fully supported; the second T4 simply idles.

After a session cutoff (idle disconnect, GPU-quota limit), just Run All again: completed runs skip themselves, the interrupted run auto-resumes from its last checkpoint, and any runs that haven't started yet train fresh. No flags or manual bookkeeping needed between sessions.

Config changes are detected automatically: if you edit a config (e.g. a different `train.lr`) after a run already trained against it, the next Run All notices the saved config no longer matches and retrains that run from scratch on its own — no flag needed. `--fresh` remains available in a cell's command to force a retrain of an UNCHANGED config (e.g. just to redo a run). `--resume` is still accepted for backward compatibility but is now a no-op — auto-resume is the default.

In [ ]:
import os, pathlib, subprocess
root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists()), None)
if root is None:  # fresh Kaggle/Colab session: not inside a clone yet, so make one
    subprocess.run(["git", "clone", "https://github.com/mtsilverstein/Megatron.git"], check=True)
    root = pathlib.Path.cwd() / "Megatron"
os.chdir(root)
print("cwd:", os.getcwd())

!git pull
!pip install -q -e .
!python -m ffmodel.data.pull --seasons 2012 2025 --out data/raw
!python -c "from pathlib import Path; from ffmodel.data.pull import pull_weekly, pull_schedules; from ffmodel.data.features import build_features; s=list(range(2012,2026)); build_features(pull_weekly(s, Path('data/raw')), pull_schedules(s, Path('data/raw'))).to_parquet('data/features_2012_2025.parquet')"
import torch; print('cuda:', torch.cuda.is_available())

## Hyperparameter sweep (optional, before committing to a walk-forward config)

Tunes ONLY on the `configs/transformer_v1_through2022.yaml` fold (train <= 2021,
val = 2022) -- held-out test seasons 2023-2025 are never touched here; the
sweep module does not import or run the eval harness at all (see
`src/ffmodel/model/sweep.py`'s module docstring for the full binding
protocol).

**Resumable for free:** if this Studio Lab session's 4h GPU quota runs out
mid-sweep, just re-run the next cell -- completed combos are skipped, the
combo that was mid-training auto-resumes from its last epoch checkpoint,
and combos that haven't started yet train fresh. No flags, no manual
bookkeeping.

**Expected wall time:** the default grid (`configs/sweep_v1.yaml`) is 24
combos; each combo is a full short training run (a handful to dozens of
epochs with early stopping), so budget on the order of a few minutes per
combo on a T4 (more for the larger `d_model`/`seq_len` combos) -- likely
more than one GPU session for the full grid. That's fine: see above.

In [ ]:
!python -m ffmodel.model.sweep --grid configs/sweep_v1.yaml --features-parquet data/features_2012_2025.parquet

In [ ]:
import json
from pathlib import Path

import pandas as pd

results = json.loads(Path("models/sweeps/v1/results.json").read_text())
# collect_results() already sorts best (lowest val_pinball) first and
# flags any not-yet-complete combo -- this is just a readable view of it.
pd.json_normalize(results)

In [ ]:
!python -m ffmodel.model.train --config configs/transformer_v1_through2022.yaml --features-parquet data/features_2012_2025.parquet

In [ ]:
!python -m ffmodel.model.train --config configs/transformer_v1_through2023.yaml --features-parquet data/features_2012_2025.parquet

In [ ]:
!python -m ffmodel.model.train --config configs/transformer_v1_through2024.yaml --features-parquet data/features_2012_2025.parquet

In [ ]:
!python -m ffmodel.model.train --config configs/transformer_v1.yaml --features-parquet data/features_2012_2025.parquet

## Promoting a sweep winner

Once `models/sweeps/v1/results.json` has a winner you're happy with (lowest
`val_pinball`, `complete: true`), turn it into the walk-forward configs the
bake-off below actually uses:

1. Copy the winning combo's `params` (from its `results.json` row) into
   **all four** `configs/transformer_v1*.yaml` files
   (`transformer_v1_through2022.yaml`, `transformer_v1_through2023.yaml`,
   `transformer_v1_through2024.yaml`, and the production
   `transformer_v1.yaml`) -- dotted keys like `model.d_model` map directly
   onto those files' `model:`/`train:` blocks. No `--fresh` needed after
   editing: the train cells above detect that each config changed and
   retrain those runs from scratch automatically on the next Run All.
2. Optionally train a small seed ensemble for the production config:
   `python -m ffmodel.model.train --config configs/transformer_v1.yaml
   --features-parquet data/features_2012_2025.parquet --seed 43`, and again
   with `--seed 44`. Each seed lands in its own sibling artifact dir
   (`v1_s43/through2025`, `v1_s44/through2025`) without touching the
   unsuffixed `v1/through2025` run.
3. To bake off (or deploy) an ensemble instead of a single seed, pass
   multiple roots:
   - Eval: repeat `--transformer-root` (e.g. `--transformer-root
     models/transformer/v1_s43 --transformer-root
     models/transformer/v1_s44`) -- the bake-off cell below shows the
     single-root form; repeat the flag to average seeds.
   - Site generator: `ffmodel.site.generate --artifact-root` accepts a
     comma-separated list (`models/transformer/v1_s43,models/transformer/v1_s44`)
     and averages them the same way -- `.github/workflows/weekly-update.yml`'s
     `ARTIFACT_ROOT` env var can be set to that comma-separated string if/when
     the deployed site should serve an ensemble instead of a single seed.

In [ ]:
import subprocess
import zipfile
from pathlib import Path

# Walk-forward bake-off: fail loudly here -- there's nothing meaningful to
# commit if this doesn't succeed.
subprocess.run(
    ["python", "-m", "ffmodel.eval.run", "--transformer-root", "models/transformer/v1",
     "--out", "models/backtests/bakeoff.json"],
    check=True,
)

subprocess.run(["git", "add", "models/transformer/v1", "models/backtests", "configs"], check=False)
subprocess.run(
    ["git", "commit", "-m", "model: transformer v1 walk-forward artifacts + bake-off results"],
    check=False,
)  # non-zero (harmlessly) if there's nothing new to commit -- don't treat as fatal

pushed = subprocess.run(["git", "push"], check=False).returncode == 0

if not pushed:
    # Kaggle/Colab-proof: those platforms often can't push with your git
    # credentials. Zip up everything a human needs to commit locally
    # instead of losing the run.
    zip_path = Path("models_artifacts.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for src_dir in ("models/transformer", "models/backtests"):
            for f in Path(src_dir).rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to("."))
        for f in Path("models/sweeps").rglob("results.json"):
            zf.write(f, f.relative_to("."))
    print(
        f"git push failed -- wrote {zip_path.resolve()} instead.\n"
        "download this file and, in your local clone:\n"
        f"  unzip {zip_path.name}\n"
        "  git add models/transformer models/backtests models/sweeps/**/results.json\n"
        '  git commit -m "model: transformer v1 walk-forward artifacts + bake-off results"\n'
        "  git push"
    )
else:
    print("pushed successfully")